# Probe health with day-level embeddings
Following tutorial here: https://oxwearables.github.io/Sensori/tutorials/day-level-health/#prepare-the-analysis-data
The pre-processing notebook: tutorials/nhanes_preprocessing.ipynb(this outputs the .npy files same as my script is curently generating)
There is also the full tutorial notebook: tutorials/nhanes_evaluation.ipynb - I am re-working this to run the linear probe against e.g. sex, BMI etc to benchmark sensori's performance on lower back sensor data


In [20]:
import json
from importlib.resources import files
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from umap import UMAP
import os

from scripts.tutorial_nhanes.prepare_nhanes_tabular import prepare_tabular
from sensori.inference import extract_embeddings # looks dodgy but does work

In [21]:
torch.cuda.is_available()

True

- first extract the embeddings - this code fetches the weights from huggingface

In [22]:
# Change this to the directory containing your preprocessed NHANES arrays.
PROCESSED_DATA_DIR = Path('/home/lucy/PycharmProjects/Sensori/data/sensori-data')
EMBEDDING_PATH = Path('mobilised_embeddings.npy')

# BATCH_SIZE = 100 needs about 70 GB of GPU memory for full-day inputs; reduce it if you run out.
BATCH_SIZE = 10
NUM_WORKERS = 6
SEED = 111

extract_embeddings(
    PROCESSED_DATA_DIR,
    output_path=EMBEDDING_PATH,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

Embeddings already exist: /home/lucy/PycharmProjects/Sensori/experiments/mobilised_embeddings.npy


PosixPath('/home/lucy/PycharmProjects/Sensori/experiments/mobilised_embeddings.npy')

## Prepare tabular data
So I think here I am prepping the demographic data to run the linear probe against

In [24]:
# df = pd.read_csv(os.path.join("..", "data", "timestamped_nep.csv"))
#
# df = df[["Local.Participant", "visit.number", "timestamp_plus_one_NEP"]]
#
# df = df.dropna(subset=["timestamp_plus_one_NEP"]).reset_index(drop=True)
#
# df["visit.number"] = (df["visit.number"]).str.upper()
#
# df["eid"] = (df["Local.Participant"].astype(str) + "_" + df["visit.number"])
#
# tabular = df[["eid", "timestamp_plus_one_NEP"]]
#
# tabular

df = pd.read_csv(os.path.join("..", "data", "demographics.csv"))

df["BMI"] = (df["weight"]/(df["height"] * df["height"])) * 10000

# df

columns = ["Local Participant", "age", "gender", "BMI", "healthstat", "smoker", "alcocons"]

tabular = df[columns]

tabular

,Local Participant,age,gender,BMI,healthstat,smoker,alcocons
0,10000,75,Male,22.491349,Average,F,4.5
1,10001,66,Male,28.293345,Average,T,0.0
2,10002,64,Female,29.470435,Average,F,1.5
3,10003,84,Male,22.340297,Good,F,3.0
4,10004,80,Male,27.909896,Bad,T,0.0
...,...,...,...,...,...,...,...
2372,33029,76,Male,23.533043,Average,T,10.5
2373,33034,58,Male,23.765432,Average,T,3.0
2374,33035,64,Male,22.229062,Very good,T,0.0
2375,33039,78,Male,23.051755,Good,T,10.5


## build participant-level embeddings
- just gonna extract a few more locally to make this match the example better
- so what's going on here... L2 normalisation and then averaging of day embeddings???
- then join to tabular (outcome) data

In [25]:
# def load_participant_embeddings(path):
#     day_embeddings = np.load(path, allow_pickle=True).item()
#     participants = {}
#     for eid, days in day_embeddings.items():
#         days = np.asarray(days, dtype=np.float32)
#         norms = np.linalg.norm(days, axis=1)
#         valid = np.isfinite(norms) & (norms > 0)  # a non-finite day gives a non-finite norm
#         if valid.any():
#             participants[str(eid)] = (days[valid] / norms[valid, None]).mean(axis=0)
#
#     frame = pd.DataFrame.from_dict(participants, orient='index')
#     frame.index.name = 'eid'
#     frame.columns = [f'embedding_{column:03d}' for column in range(frame.shape[1])]
#     return frame
#
#
# embeddings = load_participant_embeddings(EMBEDDING_PATH)
# embedding_columns = embeddings.columns.tolist()
# tabular['eid'] = tabular['eid'].astype(str)
# cohort = tabular.set_index('eid').join(embeddings, how='inner')
# print(f'Participants with tabular data and embeddings: {len(cohort):,}')
# cohort.reset_index()[['eid', 'timestamp_plus_one_NEP']].head()

def load_participant_embeddings(path):
    day_embeddings = np.load(path, allow_pickle=True).item()
    participants = {}
    for eid, days in day_embeddings.items():
        days = np.asarray(days, dtype=np.float32)
        norms = np.linalg.norm(days, axis=1)
        valid = np.isfinite(norms) & (norms > 0)
        if valid.any():
            participants[str(eid)] = (days[valid] / norms[valid, None]).mean(
                axis=0
            )

    frame = pd.DataFrame.from_dict(participants, orient="index")
    frame.index.name = "eid"  # Retains 12345_T1 as the index name
    frame.columns = [
        f"embedding_{column:03d}" for column in range(frame.shape[1])
    ]
    return frame


# 1. Load embeddings
embeddings = load_participant_embeddings(EMBEDDING_PATH)
embedding_columns = embeddings.columns.tolist()

# 2. Extract base ID for joining while preserving the original eid
embeddings = embeddings.reset_index()
embeddings["base_eid"] = embeddings["eid"].str.split("_").str[0]

# 3. Prepare demographic data
demo_cols = ["Local Participant", "age", "gender", "BMI", "healthstat", "smoker", "alcocons"]
tabular_subset = tabular[demo_cols].copy()
tabular_subset["Local Participant"] = tabular_subset[
    "Local Participant"
].astype(str)

# 4. Perform Many-to-One merge and restore 'eid' (12345_T1 format) as the index
cohort = embeddings.merge(
    tabular_subset,
    left_on="base_eid",
    right_on="Local Participant",
    how="inner",
).drop(columns=["base_eid", "Local Participant"])

# Re-index by full 'eid' to match downstream expectations
cohort = cohort.set_index("eid")

print(f"Participants with tabular data and embeddings: {len(cohort):,}")
cohort[["age", "gender", "BMI", "healthstat", "smoker", "alcocons"]].head()

Participants with tabular data and embeddings: 1,033


,age,gender,BMI,healthstat,smoker,alcocons
eid,,,,,,
10376_T1,50,Male,22.790358,Good,T,2.0
10376_T5,50,Male,22.790358,Good,T,2.0
10377_T1,43,Female,24.163265,Average,T,0.0
10378_T3,50,Female,32.046146,Average,T,1.0
10378_T5,50,Female,32.046146,Average,T,1.0


## Umap visualisation
- so I think this is just based on correlation/regression between sensori embeddings and outcome vars? there is a bunch of stuff in here which really isn't clear about the firmware etc which ig is specific to the NHANES data?
- I don't realy understand it but will try to convert it as best I can
- I guess it will say more when we actually have more than 4 datapoint....

In [13]:
# TODO: re-work this as per original sensori umap analysis

# gemini gave me this re-working of the umap chunk
# 1. Compute UMAP embeddings
# umap_table = cohort.copy()
# umap_table[['UMAP1', 'UMAP2']] = UMAP(
#     n_neighbors=50, random_state=SEED, n_jobs=1
# ).fit_transform(umap_table[embedding_columns])
#
# # 2. Filter out missing values and sample/shuffle points to avoid overplotting bias
# plot_table = (
#     umap_table.dropna(subset=['timestamp_plus_one_NEP'])
#     .sample(frac=1, random_state=SEED)
# )
#
# # 3. Define explicit colors for class 0 and class 1
# colors = {0: '#2878B5', 1: '#D95F82'}  # Blue for 0, Pink/Red for 1
#
# # 4. Create the scatter plot
# fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
#
# # Scatter plot using mapped colors
# ax.scatter(
#     plot_table['UMAP1'],
#     plot_table['UMAP2'],
#     c=plot_table['timestamp_plus_one_NEP'].map(colors),
#     s=5,
#     alpha=0.6,  # Helps visualize overlapping points
# )
#
# # Add legend entries manually
# for label, color in colors.items():
#     ax.scatter([], [], color=color, label=f'Class {label}')
#
# # 5. Styling
# ax.set_title('UMAP Projection by timestamp_plus_one_NEP (Binary)')
# ax.set(xticks=[], yticks=[], xlabel='UMAP 1', ylabel='UMAP 2')
# ax.legend(title='timestamp_plus_one_NEP')
#
# plt.show()

## encode the evaluation targets
- tbh i am slightly losing the will to live - will pick this up tomorrow.
- maybe quickly before i go i can run a quick LR or something...

In [14]:
# need to do a proper train test split
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. Extract the 5-character participant ID from the DataFrame index
group_id = df.index.astype(str).str[:5]

# 2. Set up the grouped splitter (e.g., 80% train / 20% test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# 3. Generate train/test indices using group_id
train_idx, test_idx = next(gss.split(df, groups=group_id))

# 4. Split the DataFrame using integer location (.iloc)
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Sanity Check: Verify zero overlap between train and test participants
train_participants = set(train_df.index.astype(str).str[:5])
test_participants = set(test_df.index.astype(str).str[:5])
overlap = train_participants.intersection(test_participants)

print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(
    f"Unique train participants: {len(train_participants)} | Unique test participants: {len(test_participants)}"
)
print(f"Participant overlap: {len(overlap)} (Should be 0)")

Train samples: 1901 | Test samples: 476
Unique train participants: 1901 | Unique test participants: 476
Participant overlap: 0 (Should be 0)


In [15]:
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Identify feature columns automatically
feature_cols = cohort.filter(regex=r"^embedding_").columns

# 2. Extract features and target, dropping missing values
clean_df = cohort.dropna(subset=["age"] + list(feature_cols))
X = clean_df[feature_cols]
y = clean_df["age"]

# Extract participant IDs for group-aware splitting
groups = clean_df.index.astype(str).str[:5]

# 3. Split using GroupKFold to preserve participant independence
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 4. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Fit Ridge Regression
reg = Ridge(alpha=100.0, random_state=SEED)
reg.fit(X_train_scaled, y_train)

# 6. Predict on hold-out set
y_pred = reg.predict(X_test_scaled)

# 7. Calculate evaluation metrics including Pearson's r
r_stat, p_val = pearsonr(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("--- Age Prediction Report ---")
print(f"Pearson's r: {r_stat:.3f} (p-value: {p_val:.2e})")
print(f"R² Score: {r2:.3f}")
print(f"MAE: {mae:.2f} years")
print(f"RMSE: {rmse:.2f} years")

--- Age Prediction Report ---
Pearson's r: 0.610 (p-value: 1.78e-22)
R² Score: 0.368
MAE: 7.16 years
RMSE: 8.84 years


In [16]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Identify feature columns automatically (embedding_000 through embedding_XXX)
feature_cols = cohort.filter(regex=r"^embedding_").columns

# 2. Extract features and target, dropping rows with missing values
clean_df = cohort.dropna(subset=["gender"] + list(feature_cols))
X = clean_df[feature_cols]
y = clean_df["gender"]

# Extract 5-character participant group IDs directly from the index
groups = clean_df.index.astype(str).str[:5]

# 3. Split using StratifiedGroupKFold to maintain label balance AND group independence
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(X, y, groups=groups))

# Slice features and target arrays using iloc
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 4. Scale features (essential for high-dimensional embeddings)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train Logistic Regression
clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_train_scaled, y_train)

# 6. Evaluate
y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

--- Classification Report ---
              precision    recall  f1-score   support

      Female       0.90      0.92      0.91       154
        Male       0.76      0.70      0.73        53

    accuracy                           0.86       207
   macro avg       0.83      0.81      0.82       207
weighted avg       0.86      0.86      0.86       207

ROC-AUC Score: 0.9191


In [17]:
import numpy as np
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 1. Identify feature columns automatically
feature_cols = cohort.filter(regex=r"^embedding_").columns

# 2. Extract features and target, dropping missing values
clean_df = cohort.dropna(subset=["BMI"] + list(feature_cols))
X = clean_df[feature_cols]
y = clean_df["BMI"]

# Extract participant IDs for group-aware splitting
groups = clean_df.index.astype(str).str[:5]

# 3. Split using GroupKFold to preserve participant independence
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 4. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Fit Ridge Regression
reg = Ridge(alpha=100.0, random_state=SEED)
reg.fit(X_train_scaled, y_train)

# 6. Predict on hold-out set
y_pred = reg.predict(X_test_scaled)

# 7. Calculate evaluation metrics including Pearson's r
r_stat, p_val = pearsonr(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("--- BMI Prediction Report ---")
print(f"Pearson's r: {r_stat:.3f} (p-value: {p_val:.2e})")
print(f"R² Score: {r2:.3f}")
print(f"MAE: {mae:.2f} years")
print(f"RMSE: {rmse:.2f} years")

--- BMI Prediction Report ---
Pearson's r: 0.578 (p-value: 7.88e-20)
R² Score: 0.328
MAE: 4.26 years
RMSE: 5.85 years


## matching method from sensori paper (class weighting, folds etc)
- gemini gave me some code to evaluate all metrics

In [26]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Define seed for reproducibility
SEED = 71

# 1. Identify embedding feature columns automatically
feature_cols = cohort.filter(regex=r"^embedding_").columns


# =========================================================================
# A. CONTINUOUS TARGETS: L2-Regularized Linear Regression (Ridge)
# =========================================================================
def evaluate_continuous_target(df, target_col, feature_cols, n_splits=5):
    """Fits L2-regularized Ridge regression using GroupKFold cross-validation."""
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]
    y = clean_df[target_col]
    groups = clean_df.index.astype(str).str.split("_").str[0]

    gkf = GroupKFold(n_splits=n_splits)
    oof_preds = np.zeros(len(clean_df))

    for train_idx, test_idx in gkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized linear regression
        model = Ridge(alpha=100.0, random_state=SEED)
        model.fit(X_train_scaled, y_train)

        oof_preds[test_idx] = model.predict(X_test_scaled)

    # Calculate overall dataset-level evaluation metrics
    r_stat, p_val = pearsonr(y, oof_preds)
    r2 = r2_score(y, oof_preds)
    mae = mean_absolute_error(y, oof_preds)
    rmse = root_mean_squared_error(y, oof_preds)

    print(f"=== Continuous Evaluation: {target_col} ===")
    print(f"Pearson's r : {r_stat:.3f} (p-value: {p_val:.2e})")
    print(f"R² Score    : {r2:.3f}")
    print(f"MAE         : {mae:.2f}")
    print(f"RMSE        : {rmse:.2f}\n")

    return oof_preds


# =========================================================================
# B. CATEGORICAL (BINARY & MULTI-CLASS) TARGETS: Class-Balanced L2 Logistic Regression
# =========================================================================
def evaluate_categorical_target(df, target_col, feature_cols, n_splits=5):
    """Fits class-balanced L2-regularized Logistic Regression (binary or multiclass)

    using StratifiedGroupKFold cross-validation.
    """
    clean_df = df.dropna(subset=[target_col] + list(feature_cols)).copy()
    X = clean_df[feature_cols]

    # Encode string categories into numeric labels
    le = LabelEncoder()
    y = le.fit_transform(clean_df[target_col].astype(str))
    classes = le.classes_
    num_classes = len(classes)

    # Extract base participant ID for group splitting
    groups = clean_df.index.astype(str).str.split("_").str[0]

    sgkf = StratifiedGroupKFold(
        n_splits=n_splits, shuffle=True, random_state=SEED
    )
    oof_probs = np.zeros((len(clean_df), num_classes))

    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train = y[train_idx]

        # Standardize features strictly within each training fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # L2-regularized Logistic Regression with balanced class weighting
        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=SEED,
        )
        clf.fit(X_train_scaled, y_train)

        oof_probs[test_idx] = clf.predict_proba(X_test_scaled)

    # Compute AUROC based on target cardinality
    if num_classes == 2:
        # Binary target (e.g. gender)
        overall_auroc = roc_auc_score(y, oof_probs[:, 1])
        print(
            f"=== Categorical Evaluation (Binary): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"AUROC : {overall_auroc:.4f}\n")
    else:
        # Multi-class target (e.g. healthstat)
        overall_auroc = roc_auc_score(
            y, oof_probs, multi_class="ovr", average="macro"
        )
        print(
            f"=== Categorical Evaluation (Multi-class): {target_col} [{', '.join(classes)}] ==="
        )
        print(f"Macro OVR AUROC : {overall_auroc:.4f}")

        # Per-class AUROC breakdown
        for i, cls in enumerate(classes):
            cls_binary_y = (y == i).astype(int)
            cls_auroc = roc_auc_score(cls_binary_y, oof_probs[:, i])
            print(f"  └─ Class '{cls}' AUROC: {cls_auroc:.4f}")
        print()

    return oof_probs


# -------------------------------------------------------------------------
# Run linear probe evaluation pipeline on cohort
# -------------------------------------------------------------------------
oof_age_preds = evaluate_continuous_target(cohort, "age", feature_cols)
oof_bmi_preds = evaluate_continuous_target(cohort, "BMI", feature_cols)
oof_gender_probs = evaluate_categorical_target(cohort, "gender", feature_cols)
oof_healthstat_probs = evaluate_categorical_target(
    cohort, "healthstat", feature_cols
)
oof_smoking_probs = evaluate_categorical_target(
    cohort, "smoker", feature_cols
)
oof_alcohol_preds = evaluate_continuous_target(cohort, "alcocons", feature_cols)


=== Continuous Evaluation: age ===
Pearson's r : 0.663 (p-value: 1.06e-131)
R² Score    : 0.430
MAE         : 6.67
RMSE        : 8.20

=== Continuous Evaluation: BMI ===
Pearson's r : 0.597 (p-value: 1.10e-100)
R² Score    : 0.334
MAE         : 3.83
RMSE        : 5.15

=== Categorical Evaluation (Binary): gender [Female, Male] ===
AUROC : 0.9038

=== Categorical Evaluation (Multi-class): healthstat [Average, Bad, Good, Less than average, Very bad, Very good] ===
Macro OVR AUROC : 0.6414
  └─ Class 'Average' AUROC: 0.5081
  └─ Class 'Bad' AUROC: 0.5857
  └─ Class 'Good' AUROC: 0.5730
  └─ Class 'Less than average' AUROC: 0.4841
  └─ Class 'Very bad' AUROC: 0.8618
  └─ Class 'Very good' AUROC: 0.8354

=== Categorical Evaluation (Binary): smoker [F, T] ===
AUROC : 0.5870

=== Continuous Evaluation: alcocons ===
Pearson's r : 0.035 (p-value: 2.64e-01)
R² Score    : -0.246
MAE         : 5.24
RMSE        : 7.17

